<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V18_CrossCity_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V18 — Cross-City Transfer Learning: Anomaly Detection in BSS-Nachfrage

**Ziel:** Übertragbarkeit von Anomalieerkennungswissen zwischen Städten untersuchen.

## Experimente:
1. **Transfer-Modus** (MA→HD): Target-Only vs. Zero-Shot vs. Fine-Tune 1%
2. **Datenmenge** (MA→HD): Fine-Tune 1% / 5% / 20%
3. **Source-City-Wahl** (GI→HD): Alternative Source-Stadt
4. **Multi-Source** (MA+GI+KL→HD): Mehrere Source-Städte

## Finalisten aus V17:
- `EXP0_Baseline_LSTM_AE` — Rekonstruktionsbasiert
- `EXP3d_Forecast_LSTM_dir_ZNorm` — Vorhersagebasiert (directional scoring)

## Fokus: Punkt-Anomalien (Spike/Drop) + Kollektive Anomalien


In [ ]:
# ══════════════════════════════════════════════════════════════
# 0a — Colab Setup
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ══════════════════════════════════════════════════════════════
# 0b — Imports
# ══════════════════════════════════════════════════════════════
import os, math, json, random, warnings, time, re, glob, pickle
from dataclasses import dataclass, field, asdict
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_auc_score, f1_score, precision_score, recall_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")


TF: 2.19.0, GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ══════════════════════════════════════════════════════════════
# 0c — V18 Config + Pfade
# ══════════════════════════════════════════════════════════════
DATA_BASE    = '/content/drive/MyDrive/BA_Colab/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

RUN_NAME    = 'v18_transfer_learning'
RESULTS_DIR = f'{DATA_BASE}/{RUN_NAME}'
MODELS_DIR  = f'{RESULTS_DIR}/models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# V17 Modelle (Source-Modelle von Mannheim)
V17_DIR = f'{DATA_BASE}/v17_ad_improvements/models'
V17_AE_PATH = f'{V17_DIR}/baseline_lstm_ae.keras'
V17_FC_PATH = f'{V17_DIR}/lstm_forecaster.keras'
V17_SCALER_PATH = f'{V17_DIR}/scaler_mannheim.pkl'

# Daten-Pfade: alle Staedte
CITY_DEMAND_PATHS = {
    "Mannheim":       f'{CLEANED_BASE}/demand/Mannheim/demand_cleaned.parquet',
    "Heidelberg":     f'{CLEANED_BASE}/demand/Heidelberg/demand_cleaned.parquet',
    "Giessen":        f'{CLEANED_BASE}/demand/Giessen/demand_cleaned.parquet',
    "Kaiserslautern": f'{CLEANED_BASE}/demand/Kaiserslautern/demand_cleaned.parquet',
}
GEO_PATH           = f'{CLEANED_BASE}/geo_information/geo_information.parquet'
STATION_NAMES_PATH = f'{DATA_BASE}/station_names/station_names.parquet'

# Results-Cache (ueberlebt Laufzeit-Unterbrechungen)
RESULTS_CACHE_PATH = f'{RESULTS_DIR}/v18_results_cache.json'

# V18 Modell-Dateien (Checkpoints fuer alle Trainings)
V18_MODEL_FILES = {
    # Exp 1+2: AE
    "E1_AE_HD_1pct":       f'{MODELS_DIR}/e1_ae_hd_target_only_1pct.keras',
    "E1_AE_HD_100pct":     f'{MODELS_DIR}/e1_ae_hd_target_only_100pct.keras',
    "E2_AE_FT_MA_HD_1pct": f'{MODELS_DIR}/e2_ae_ft_ma_hd_1pct.keras',
    "E2_AE_FT_MA_HD_5pct": f'{MODELS_DIR}/e2_ae_ft_ma_hd_5pct.keras',
    "E2_AE_FT_MA_HD_20pct":f'{MODELS_DIR}/e2_ae_ft_ma_hd_20pct.keras',
    # Exp 1+2: FC
    "E1_FC_HD_1pct":       f'{MODELS_DIR}/e1_fc_hd_target_only_1pct.keras',
    "E1_FC_HD_100pct":     f'{MODELS_DIR}/e1_fc_hd_target_only_100pct.keras',
    "E2_FC_FT_MA_HD_1pct": f'{MODELS_DIR}/e2_fc_ft_ma_hd_1pct.keras',
    "E2_FC_FT_MA_HD_5pct": f'{MODELS_DIR}/e2_fc_ft_ma_hd_5pct.keras',
    "E2_FC_FT_MA_HD_20pct":f'{MODELS_DIR}/e2_fc_ft_ma_hd_20pct.keras',
    # Exp 3: Giessen
    "E3_FC_GI_source":     f'{MODELS_DIR}/e3_fc_gi_source.keras',
    "E3_FC_FT_GI_HD_1pct": f'{MODELS_DIR}/e3_fc_ft_gi_hd_1pct.keras',
    # Exp 4: Multi-Source
    "E4_FC_multi_source":  f'{MODELS_DIR}/e4_fc_multi_source.keras',
    "E4_FC_FT_multi_1pct": f'{MODELS_DIR}/e4_fc_ft_multi_1pct.keras',
    "E4_FC_FT_multi_10pct":f'{MODELS_DIR}/e4_fc_ft_multi_10pct.keras',
}

# Pruefen: V17 Modelle vorhanden?
for name, path in [("V17 AE", V17_AE_PATH), ("V17 FC", V17_FC_PATH), ("V17 Scaler", V17_SCALER_PATH)]:
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'} {name}: {os.path.basename(path)}")

# Force-Retrain Flag (auf True setzen um alle Modelle neu zu trainieren)
FORCE_RETRAIN = False
print(f"\nFORCE_RETRAIN = {FORCE_RETRAIN}")
print(f"Results: {RESULTS_DIR}")


  ✓ V17 AE: baseline_lstm_ae.keras
  ✓ V17 FC: lstm_forecaster.keras
  ✓ V17 Scaler: scaler_mannheim.pkl

FORCE_RETRAIN = False
Results: /content/drive/MyDrive/BA_Colab/data/v18_transfer_learning


In [ ]:
# ══════════════════════════════════════════════════════════════
# 0d — V18 Config (erweitert V17)
# ══════════════════════════════════════════════════════════════
@dataclass
class V18Config:
    # --- Daten-Pipeline (identisch V17) ---
    aggregation_minutes: int = 60
    train_ratio: float = 0.67
    val_ratio: float = 0.82
    min_events_per_day: float = 3.0
    rolling_days: int = 56
    min_context_obs: int = 20

    # --- AE / FC Architektur (identisch V17) ---
    ae_window_size: int = 24
    ae_latent_dim: int = 32
    ae_layers: int = 2
    ae_dropout: float = 0.10
    ae_epochs: int = 50
    ae_batch_size: int = 2048
    ae_lr: float = 1e-3
    ae_early_stop: int = 8

    # --- Fine-Tuning (NEU in V18) ---
    ft_epochs: int = 30
    ft_lr: float = 1e-4
    ft_batch_size: int = 512
    ft_early_stop: int = 5

    # --- Threshold / Labeling ---
    low_demand_q: float = 0.33
    high_demand_q: float = 0.67

    # --- Pipeline ---
    require_contiguous: bool = True
    use_bidirectional: bool = False

    # --- Features (identisch V17) ---
    ae_features: List[str] = field(default_factory=lambda: [
        "n_lends", "n_returns",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos",
        "month_sin", "month_cos", "is_weekend"
    ])
    ae_score_features: List[str] = field(default_factory=lambda: ["n_lends", "n_returns"])

    # --- Injection (identisch V17) ---
    injection_rate: float = 0.015
    injection_seed: int = 42
    injection_scenario: str = "balanced"

    # --- Staedte ---
    source_city: str = "Mannheim"
    target_city: str = "Heidelberg"
    alt_source: str = "Giessen"
    multi_sources: List[str] = field(
        default_factory=lambda: ["Mannheim", "Giessen", "Kaiserslautern"])

cfg = V18Config()
print(cfg)


V18Config(aggregation_minutes=60, train_ratio=0.67, val_ratio=0.82, min_events_per_day=3.0, rolling_days=56, min_context_obs=20, ae_window_size=24, ae_latent_dim=32, ae_layers=2, ae_dropout=0.1, ae_epochs=50, ae_batch_size=2048, ae_lr=0.001, ae_early_stop=8, ft_epochs=30, ft_lr=0.0001, ft_batch_size=512, ft_early_stop=5, low_demand_q=0.33, high_demand_q=0.67, require_contiguous=True, use_bidirectional=False, ae_features=['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend'], ae_score_features=['n_lends', 'n_returns'], injection_rate=0.015, injection_seed=42, injection_scenario='balanced', source_city='Mannheim', target_city='Heidelberg', alt_source='Giessen', multi_sources=['Mannheim', 'Giessen', 'Kaiserslautern'])


---\n## Shared Functions (aus V17 übernommen)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1 — Aggregation
# ══════════════════════════════════════════════════════════════
def aggregate_station_level(df: pd.DataFrame, minutes: int = 60) -> pd.DataFrame:
    """
    Aggregiert Rohdaten auf Stations-Zeitbin-Ebene.
    Erwartetes Schema:
      - location_id
      - timestamp
      - network_name
      - station_name_id
      - station_id
      - n_lends
      - n_returns
    """
    df = df.copy()

    required_cols = {
        "timestamp", "station_id", "station_name_id", "n_lends", "n_returns"
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"aggregate_station_level: Fehlende Spalten: {sorted(missing)}")

    # Typen robust setzen
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp", "station_id"]).copy()

    df["n_lends"] = pd.to_numeric(df["n_lends"], errors="coerce").fillna(0).astype(np.int32)
    df["n_returns"] = pd.to_numeric(df["n_returns"], errors="coerce").fillna(0).astype(np.int32)

    # Ziel-Zeitbin
    df["hour_ts"] = df["timestamp"].dt.floor(f"{minutes}min")

    # WICHTIG:
    # Nur auf station_id + hour_ts aggregieren.
    # NICHT station_name_id in den Groupby-Key aufnehmen, sonst entstehen
    # später leicht doppelte (station_id, hour_ts)-Kombinationen.
    agg_dict = {
        "n_lends": ("n_lends", "sum"),
        "n_returns": ("n_returns", "sum"),
        "station_name_id": ("station_name_id", "first"),
    }

    if "station_name" in df.columns:
        agg_dict["station_name"] = ("station_name", "first")
    if "latitude" in df.columns:
        agg_dict["latitude"] = ("latitude", "first")
    if "longitude" in df.columns:
        agg_dict["longitude"] = ("longitude", "first")
    if "location_id" in df.columns:
        agg_dict["location_id"] = ("location_id", "first")
    if "network_name" in df.columns:
        agg_dict["network_name"] = ("network_name", "first")

    agg = (
        df.groupby(["station_id", "hour_ts"], as_index=False)
          .agg(**agg_dict)
          .sort_values(["station_id", "hour_ts"])
          .reset_index(drop=True)
    )

    # Sicherheitscheck: es darf jetzt keine Duplikate mehr geben
    dup_mask = agg.duplicated(subset=["station_id", "hour_ts"], keep=False)
    if dup_mask.any():
        dup_n = int(dup_mask.sum())
        raise ValueError(
            f"aggregate_station_level: Nach Aggregation existieren noch {dup_n} "
            f"doppelte Zeilen auf ['station_id', 'hour_ts']."
        )

    # Abgeleitete Zielgrößen
    agg["total_demand"] = agg["n_lends"] + agg["n_returns"]
    agg["net_flow"] = agg["n_returns"] - agg["n_lends"]
    agg["abs_net_flow"] = agg["net_flow"].abs()

    # Zyklische Zeit-Features
    h = agg["hour_ts"].dt.hour + agg["hour_ts"].dt.minute / 60.0
    d = agg["hour_ts"].dt.dayofweek
    m = agg["hour_ts"].dt.month

    agg["hour_sin"] = np.sin(2 * np.pi * h / 24)
    agg["hour_cos"] = np.cos(2 * np.pi * h / 24)
    agg["dow_sin"] = np.sin(2 * np.pi * d / 7)
    agg["dow_cos"] = np.cos(2 * np.pi * d / 7)
    agg["month_sin"] = np.sin(2 * np.pi * m / 12)
    agg["month_cos"] = np.cos(2 * np.pi * m / 12)
    agg["is_weekend"] = (d >= 5).astype(float)

    return agg

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2 — Gap-Fill
# ══════════════════════════════════════════════════════════════
def fill_missing_time_bins(x: pd.DataFrame, minutes: int = 60) -> pd.DataFrame:
    """
    Fuellt fehlende Zeitbins je Station auf.
    Erwartet eindeutig eindeutige Kombinationen aus (station_id, hour_ts).
    Falls dennoch Duplikate vorhanden sind, werden sie vorher defensiv aggregiert.
    """
    x = x.copy()

    required_cols = {"station_id", "hour_ts"}
    missing = required_cols - set(x.columns)
    if missing:
        raise ValueError(f"fill_missing_time_bins: Fehlende Spalten: {sorted(missing)}")

    x["hour_ts"] = pd.to_datetime(x["hour_ts"], utc=True, errors="coerce")
    x = x.dropna(subset=["station_id", "hour_ts"]).copy()

    # Defensive Duplikatbehandlung für den Fall, dass upstream doch etwas schiefgeht
    dup_mask = x.duplicated(subset=["station_id", "hour_ts"], keep=False)
    if dup_mask.any():
        print(f"  WARNUNG: {int(dup_mask.sum()):,} duplizierte Zeilen vor Gap-Fill gefunden. "
              f"Sie werden auf (station_id, hour_ts) zusammengefasst.")

        agg_dict = {}
        for col in ["n_lends", "n_returns", "total_demand", "net_flow", "abs_net_flow"]:
            if col in x.columns:
                agg_dict[col] = (col, "sum")

        for col in ["station_name_id", "station_name", "latitude", "longitude", "location_id", "network_name"]:
            if col in x.columns:
                agg_dict[col] = (col, "first")

        x = (
            x.groupby(["station_id", "hour_ts"], as_index=False)
             .agg(**agg_dict)
             .sort_values(["station_id", "hour_ts"])
             .reset_index(drop=True)
        )

    full_range = pd.date_range(
        start=x["hour_ts"].min(),
        end=x["hour_ts"].max(),
        freq=f"{minutes}min",
        tz=x["hour_ts"].dt.tz
    )

    stations = x["station_id"].dropna().unique()
    idx = pd.MultiIndex.from_product(
        [stations, full_range],
        names=["station_id", "hour_ts"]
    )

    x = (
        x.set_index(["station_id", "hour_ts"])
         .sort_index()
         .reindex(idx)
    )

    # Station-Metadaten forward/backward füllen
    meta_cols = ["station_name_id", "station_name", "latitude", "longitude", "location_id", "network_name"]
    for col in meta_cols:
        if col in x.columns:
            x[col] = x.groupby(level=0)[col].ffill().bfill()

    # Demand-Spalten auf 0 setzen
    demand_cols = ["n_lends", "n_returns", "total_demand", "net_flow", "abs_net_flow"]
    for col in demand_cols:
        if col in x.columns:
            x[col] = x[col].fillna(0)

    x = x.reset_index()

    # Falls total_demand/net_flow noch nicht vorhanden oder durch Reindex inkonsistent
    if "n_lends" in x.columns and "n_returns" in x.columns:
        x["total_demand"] = x["n_lends"] + x["n_returns"]
        x["net_flow"] = x["n_returns"] - x["n_lends"]
        x["abs_net_flow"] = x["net_flow"].abs()

    # Zeitfeatures neu berechnen
    h = x["hour_ts"].dt.hour + x["hour_ts"].dt.minute / 60.0
    d = x["hour_ts"].dt.dayofweek
    m = x["hour_ts"].dt.month

    x["hour_sin"] = np.sin(2 * np.pi * h / 24)
    x["hour_cos"] = np.cos(2 * np.pi * h / 24)
    x["dow_sin"] = np.sin(2 * np.pi * d / 7)
    x["dow_cos"] = np.cos(2 * np.pi * d / 7)
    x["month_sin"] = np.sin(2 * np.pi * m / 12)
    x["month_cos"] = np.cos(2 * np.pi * m / 12)
    x["is_weekend"] = (d >= 5).astype(float)

    return x

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3 — Station-Filter, Demand-Regime, Train/Val/Test Split
# ══════════════════════════════════════════════════════════════
def prepare_stations_and_splits(df: pd.DataFrame, cfg, city_name=""):
    n_days = (df["hour_ts"].max() - df["hour_ts"].min()).days + 1
    min_events = int(n_days * cfg.min_events_per_day)
    station_volume = df.groupby("station_id")["total_demand"].sum()
    active_ids = station_volume[station_volume >= min_events].index.tolist()
    df = df[df["station_id"].isin(active_ids)].copy()

    station_profile = (
        df.groupby(["station_id", "station_name"], as_index=False)
          .agg(avg_total_demand_h=("total_demand", "mean"),
               avg_lends_h=("n_lends", "mean"),
               avg_returns_h=("n_returns", "mean"),
               latitude=("latitude", "first"),
               longitude=("longitude", "first"))
    )
    q1 = station_profile["avg_total_demand_h"].quantile(cfg.low_demand_q)
    q2 = station_profile["avg_total_demand_h"].quantile(cfg.high_demand_q)
    station_profile["demand_regime"] = station_profile["avg_total_demand_h"].apply(
        lambda x: "low" if x <= q1 else ("mid" if x <= q2 else "high"))
    df = df.merge(station_profile[["station_id", "demand_regime", "avg_total_demand_h"]],
                  on="station_id", how="left")

    t_min, t_max = df["hour_ts"].min(), df["hour_ts"].max()
    total_h = (t_max - t_min).total_seconds() / 3600
    train_end = t_min + pd.Timedelta(hours=int(total_h * cfg.train_ratio))
    val_end   = t_min + pd.Timedelta(hours=int(total_h * cfg.val_ratio))

    cn = f" ({city_name})" if city_name else ""
    print(f"  Aktive Stationen{cn}: {df['station_id'].nunique()}")
    print(f"  Regime: {station_profile['demand_regime'].value_counts().to_dict()}")
    print(f"  Zeitraum: {t_min.date()} bis {t_max.date()} ({n_days} Tage)")
    print(f"  Split: Train < {train_end.date()}, Val < {val_end.date()}, Test ab {val_end.date()}")
    return df, station_profile, train_end, val_end


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4 — Statistik-Pipeline (Kontext-Scores, Poisson, ECDF, Labels)
# ══════════════════════════════════════════════════════════════
TARGETS = ["n_lends", "n_returns", "total_demand", "net_flow"]
COUNT_TARGETS = ["n_lends", "n_returns", "total_demand"]

def add_context_keys(df):
    df = df.copy()
    df["hour"] = df["hour_ts"].dt.hour
    df["dow"]  = df["hour_ts"].dt.dayofweek
    df["is_wknd"] = (df["dow"] >= 5).astype(int)
    return df

def rolling_context_scores_vectorized(df, target, rolling_days=56, min_obs=20):
    df = df.copy().sort_values(["station_id","hour_ts"])
    window_h = rolling_days * 24
    g = df.groupby(["station_id","hour","is_wknd"])[target]
    roll_mean = g.transform(lambda s: s.rolling(window_h, min_periods=min_obs).mean())
    roll_std  = g.transform(lambda s: s.rolling(window_h, min_periods=min_obs).std())
    roll_std  = roll_std.clip(lower=1e-6)
    df[f"{target}_z_ctx_roll"] = (df[target] - roll_mean) / roll_std
    return df

def add_rolling_poisson_scores_vectorized(df, target, rolling_days=56, min_obs=20):
    from scipy.stats import poisson
    df = df.copy().sort_values(["station_id","hour_ts"])
    window_h = rolling_days * 24
    g = df.groupby(["station_id","hour","is_wknd"])[target]
    roll_mean = g.transform(lambda s: s.rolling(window_h, min_periods=min_obs).mean())
    lam = roll_mean.clip(lower=0.01).values
    vals = df[target].values.astype(float)
    upper = 1.0 - poisson.cdf(vals - 1, lam)
    lower = poisson.cdf(vals, lam)
    df[f"{target}_poisson_upper_score"] = -np.log10(np.clip(upper, 1e-15, 1.0))
    df[f"{target}_poisson_lower_score"] = -np.log10(np.clip(lower, 1e-15, 1.0))
    return df

def percentile_score(train_vals, all_vals):
    from scipy.stats import percentileofscore
    train_sorted = np.sort(train_vals)
    return np.array([percentileofscore(train_sorted, v, kind='rank')/100 for v in all_vals])

def calibrate_scores_by_station(df, train_mask, score_cols):
    df = df.copy()
    for col in score_cols:
        cal_col = f"{col}_cal"
        df[cal_col] = 0.5
        for sid in df["station_id"].unique():
            sm = df["station_id"] == sid
            train_vals = df.loc[sm & train_mask, col].dropna().values
            if len(train_vals) < 10:
                train_vals = df.loc[train_mask, col].dropna().values
            all_vals = df.loc[sm, col].values
            df.loc[sm, cal_col] = percentile_score(train_vals, all_vals)
    return df

def build_eval_labels_calibrated(df, upper_thresh=0.995, lower_thresh=0.005):
    df = df.copy()
    cal_cols_upper = [c for c in df.columns if c.endswith("_score_upper_cal")]
    cal_cols_lower = [c for c in df.columns if c.endswith("_score_lower_cal")]
    cal_cols_pu    = [c for c in df.columns if c.endswith("_poisson_upper_score_cal")]
    cal_cols_pl    = [c for c in df.columns if c.endswith("_poisson_lower_score_cal")]
    is_high = pd.DataFrame(False, index=df.index, columns=["any"])
    for col in cal_cols_upper + cal_cols_pu:
        is_high["any"] |= (df[col] >= upper_thresh)
    is_low = pd.DataFrame(False, index=df.index, columns=["any"])
    for col in cal_cols_lower + cal_cols_pl:
        is_low["any"] |= (df[col] <= lower_thresh)
    df["label_eval"] = "normal"
    df.loc[is_high["any"], "label_eval"] = "anomal_high"
    df.loc[is_low["any"], "label_eval"] = "anomal_low"
    return df

def run_statistical_pipeline(df, cfg, train_end):
    df = add_context_keys(df)
    for tgt in TARGETS:
        df = rolling_context_scores_vectorized(df, target=tgt,
            rolling_days=cfg.rolling_days, min_obs=cfg.min_context_obs)
    for tgt in TARGETS:
        zc = f"{tgt}_z_ctx_roll"
        df[f"{tgt}_score_upper"] = df[zc]
        df[f"{tgt}_score_lower"] = -df[zc]
    for tgt in COUNT_TARGETS:
        df = add_rolling_poisson_scores_vectorized(df, target=tgt,
            rolling_days=cfg.rolling_days, min_obs=cfg.min_context_obs)
    raw_score_cols = []
    for tgt in TARGETS:
        raw_score_cols += [f"{tgt}_score_upper", f"{tgt}_score_lower"]
    for tgt in COUNT_TARGETS:
        raw_score_cols += [f"{tgt}_poisson_upper_score", f"{tgt}_poisson_lower_score"]
    train_mask = df["hour_ts"] < train_end
    df = calibrate_scores_by_station(df, train_mask, raw_score_cols)
    df = build_eval_labels_calibrated(df)
    anomaly_rate = df["label_eval"].isin(["anomal_high", "anomal_low"]).mean()
    print(f"  Label: {df['label_eval'].value_counts().to_dict()}, Rate: {anomaly_rate:.4f}")
    return df


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5 — Sequenzbuilder (Window-Level Labels)
# ══════════════════════════════════════════════════════════════
from numpy.lib.stride_tricks import sliding_window_view

def make_sequences_with_window_labels(
    x: pd.DataFrame, feature_cols: List[str], window_size: int,
    synth_label_col: str = "synth_label",
    synth_type_col: str = "synth_type",
    require_contiguous: bool = True,
    agg_minutes: int = 60
) -> Tuple[np.ndarray, pd.DataFrame]:
    X_parts, meta_parts = [], []
    expected_ns = pd.Timedelta(minutes=agg_minutes).value

    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").reset_index(drop=True)
        n = len(g)
        if n < window_size:
            continue

        vals = g[feature_cols].to_numpy(dtype=np.float32)
        win  = sliding_window_view(vals, window_shape=window_size, axis=0)
        win  = np.moveaxis(win, -1, 1)

        valid_mask = np.ones(n - window_size + 1, dtype=bool)
        if require_contiguous:
            ts_int = pd.to_datetime(g["hour_ts"]).astype("int64").to_numpy()
            diffs  = np.diff(ts_int)
            step_ok = (diffs == expected_ns).astype(np.int8)
            if window_size > 1:
                ok_sums = np.convolve(step_ok, np.ones(window_size-1, dtype=np.int32), mode="valid")
                valid_mask = (ok_sums == (window_size - 1))

        if not valid_mask.any():
            continue

        end_indices = np.arange(window_size - 1, n)[valid_mask]
        X_parts.append(win[valid_mask])

        meta_chunk = g.iloc[end_indices].copy()
        synth_arr = g[synth_label_col].to_numpy()
        type_arr  = g[synth_type_col].to_numpy()

        window_labels, window_types, window_counts = [], [], []
        for end_idx in end_indices:
            start_idx = end_idx - window_size + 1
            w_synth = synth_arr[start_idx:end_idx + 1]
            w_type  = type_arr[start_idx:end_idx + 1]
            has_synth = int(w_synth.max())
            if has_synth:
                synth_pos = np.where(w_synth == 1)[0]
                wtype = w_type[synth_pos[-1]]
            else:
                wtype = "none"
            window_labels.append(has_synth)
            window_types.append(wtype)
            window_counts.append(int(w_synth.sum()))

        meta_chunk["synth_label"] = window_labels
        meta_chunk["synth_type"]  = window_types
        meta_chunk["n_synth_in_window"] = window_counts
        meta_parts.append(meta_chunk)

    if X_parts:
        X    = np.concatenate(X_parts, axis=0)
        meta = pd.concat(meta_parts, axis=0).reset_index(drop=True)
    else:
        X    = np.empty((0, window_size, len(feature_cols)), dtype=np.float32)
        meta = pd.DataFrame()
    return X, meta


In [ ]:
# ══════════════════════════════════════════════════════════════
# 6 — Modell-Architekturen (identisch V17)
# ══════════════════════════════════════════════════════════════
def build_lstm_autoencoder(
    n_input_features, window_size, latent_dim=32, n_layers=2,
    dropout=0.1, bidirectional=False, n_output_features=None
):
    n_output_features = n_output_features or n_input_features
    inputs = keras.Input(shape=(window_size, n_input_features), name="encoder_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        lstm = layers.LSTM(latent_dim, return_sequences=return_sequences,
                           dropout=dropout, name=f"encoder_lstm_{i+1}")
        x = layers.Bidirectional(lstm, name=f"encoder_bi_{i+1}")(x) if bidirectional else lstm(x)

    latent = x
    x = layers.RepeatVector(window_size, name="repeat_vector")(latent)

    for i in range(n_layers):
        lstm = layers.LSTM(latent_dim, return_sequences=True,
                           dropout=dropout, name=f"decoder_lstm_{i+1}")
        x = layers.Bidirectional(lstm, name=f"decoder_bi_{i+1}")(x) if bidirectional else lstm(x)

    outputs = layers.TimeDistributed(layers.Dense(n_output_features), name="output_dense")(x)
    return keras.Model(inputs, outputs, name="lstm_autoencoder")


def build_lstm_forecaster(
    n_input_features, n_output_features, context_steps=23,
    latent_dim=32, n_layers=2, dropout=0.1
):
    inputs = keras.Input(shape=(context_steps, n_input_features), name="forecast_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        x = layers.LSTM(latent_dim, return_sequences=return_sequences,
                        dropout=dropout, name=f"forecast_lstm_{i+1}")(x)
    outputs = layers.Dense(n_output_features, name="forecast_output")(x)
    return keras.Model(inputs, outputs, name="lstm_forecaster")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 7 — Training + Caching Helpers
# ══════════════════════════════════════════════════════════════
def train_model_generic(model, X_train, Y_train, X_val, Y_val,
                        epochs, lr, batch_size, early_stop, verbose=1):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0), loss="mse")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=early_stop,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1)
    ]
    history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
                        epochs=epochs, batch_size=batch_size,
                        callbacks=callbacks, verbose=verbose)
    return model, history


def load_or_train(model_path, build_fn, force=False):
    """Lade Modell aus Cache oder trainiere neu. Speichert automatisch."""
    if os.path.exists(model_path) and not force:
        print(f"  ✓ Lade aus Cache: {os.path.basename(model_path)}")
        model = keras.models.load_model(model_path)
        return model, None  # kein History bei Cache
    print(f"  ▶ Trainiere neu: {os.path.basename(model_path)}")
    model, history = build_fn()
    model.save(model_path)
    print(f"  ✓ Gespeichert: {os.path.basename(model_path)}")
    return model, history


In [ ]:
# ══════════════════════════════════════════════════════════════
# 8 — Scoring Helpers (Reconstruction, Forecast, Z-Norm)
# ══════════════════════════════════════════════════════════════
def predict_in_batches(model, X, batch_size=2048):
    preds = []
    for i in range(0, len(X), batch_size):
        preds.append(model.predict(X[i:i+batch_size], verbose=0))
    return np.concatenate(preds, axis=0)


def compute_reconstruction_scores(X_true, X_pred, ae_features, score_features, mode="last_step_mean"):
    """Reconstruction Error nur auf score_features, letzter Zeitschritt."""
    score_idx = [ae_features.index(f) for f in score_features]
    if mode == "last_step_mean":
        err = (X_true[:, -1, :][:, score_idx] - X_pred[:, -1, :][:, score_idx]) ** 2
        return err.mean(axis=1)
    elif mode == "full_mean":
        err = (X_true[:, :, score_idx] - X_pred[:, :, score_idx]) ** 2
        return err.mean(axis=(1, 2))
    raise ValueError(f"Unknown mode: {mode}")


def compute_forecast_scores(model, X_input, X_actual_last_step,
                            score_features, ae_features, mode="directional"):
    X_pred = predict_in_batches(model, X_input)
    score_idx = [ae_features.index(f) for f in score_features]
    X_actual = X_actual_last_step[:, score_idx]
    err = (X_pred - X_actual) ** 2
    if mode == "mean":
        return err.mean(axis=1)
    elif mode == "directional":
        abs_err = np.abs(X_pred - X_actual)
        denom = np.maximum(np.maximum(np.abs(X_actual), np.abs(X_pred)), 1e-6)
        rel_err = abs_err / denom
        return 0.5 * abs_err.mean(axis=1) + 0.5 * rel_err.mean(axis=1)
    raise ValueError(f"Unknown mode: {mode}")


def znorm_score_by_hour(meta_df, raw_score_col, val_start, test_start, new_col="score_znorm"):
    """Z-Normalisierung pro Stunde, kalibriert auf VAL-Normaledaten."""
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) & (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    mu_h, std_h = {}, {}
    for h in range(24):
        vals = meta.loc[val_normal & (hour_col == h), raw_score_col].dropna()
        mu_h[h]  = vals.mean() if len(vals) > 0 else 0.0
        std_h[h] = vals.std()  if len(vals) > 1 else 1.0

    hours = hour_col.values
    raw = meta[raw_score_col].values
    znorm = np.zeros_like(raw)
    for i in range(len(raw)):
        znorm[i] = (raw[i] - mu_h[hours[i]]) / max(std_h[hours[i]], 1e-8)

    meta[new_col] = znorm
    return meta, mu_h, std_h


In [ ]:
# ══════════════════════════════════════════════════════════════
# 9 — Synthetic Injection (aus V17, nur auf Target-Stadt!)
# ══════════════════════════════════════════════════════════════
INJECTION_SCENARIOS = {
    "balanced": {
        "desc": "Gleichverteilung ohne Collective-Dominanz",
        "probs": [0.35, 0.30, 0.25, 0.10],
        "collective_block_len": (2, 4),
    },
}

def inject_synthetic_anomalies(df, test_start, scenario="balanced",
                                injection_rate=0.015, seed=42, verbose=True):
    scn = INJECTION_SCENARIOS[scenario]
    probs = scn["probs"]
    block_min, block_max = scn["collective_block_len"]
    rng = np.random.RandomState(seed)
    out = df.copy()
    out["original_n_lends"]   = out["n_lends"].copy()
    out["original_n_returns"] = out["n_returns"].copy()
    out["synth_label"] = 0
    out["synth_type"]  = "none"

    test_mask = out["hour_ts"] >= test_start
    test_idx  = out[test_mask].index.to_numpy()
    if len(test_idx) == 0:
        print("  WARNUNG: Keine Test-Daten!"); return out

    n_inject = max(1, int(len(test_idx) * injection_rate))
    has_demand = out.loc[test_idx][out.loc[test_idx, "total_demand"] > 0].index.to_numpy()
    if len(has_demand) < n_inject: n_inject = len(has_demand)

    inject_idx = rng.choice(has_demand, size=n_inject, replace=False)
    types = rng.choice(["point_spike","point_drop","contextual","collective"],
                       size=n_inject, p=probs)
    injected_indices = set()

    for idx, anom_type in zip(inject_idx, types):
        if idx in injected_indices: continue
        row = out.loc[idx]

        if anom_type == "point_spike":
            factor = rng.uniform(3.0, 8.0)
            out.loc[idx, "n_lends"]   = int(row["n_lends"] * factor) + rng.randint(2, 10)
            out.loc[idx, "n_returns"] = int(row["n_returns"] * factor) + rng.randint(2, 10)
            out.loc[idx, "synth_label"] = 1
            out.loc[idx, "synth_type"]  = "point_spike"
            injected_indices.add(idx)

        elif anom_type == "point_drop":
            regime = row.get("demand_regime", "low")
            if (regime == "high" and row["total_demand"] >= 8) or \
               (regime == "mid" and row["total_demand"] >= 5):
                out.loc[idx, "n_lends"]   = rng.randint(0, 2)
                out.loc[idx, "n_returns"] = rng.randint(0, 2)
                out.loc[idx, "synth_label"] = 1
                out.loc[idx, "synth_type"]  = "point_drop"
                injected_indices.add(idx)

        elif anom_type == "contextual":
            # Kontextuelle werden injiziert aber NICHT ausgewertet in V18
            hour = row["hour_ts"].hour
            sid  = row["station_id"]
            same_st = out[(out["station_id"]==sid) & test_mask & (~out.index.isin(injected_indices))]
            if hour in [7,8,9,17,18,19]:
                contrast = same_st[same_st["hour_ts"].dt.hour.isin([1,2,3,4])]
            else:
                contrast = same_st[same_st["hour_ts"].dt.hour.isin([8,9,17,18])]
            if len(contrast) > 0:
                swap_idx = contrast.index[rng.randint(0, len(contrast))]
                out.loc[idx, "n_lends"]   = out.loc[swap_idx, "original_n_lends"]
                out.loc[idx, "n_returns"] = out.loc[swap_idx, "original_n_returns"]
                out.loc[idx, "synth_label"] = 1
                out.loc[idx, "synth_type"]  = "contextual"
                injected_indices.add(idx)

        elif anom_type == "collective":
            block_len = rng.randint(block_min, block_max + 1)
            sid = row["station_id"]
            station_test = out[(out["station_id"]==sid) & test_mask].sort_values("hour_ts")
            if idx in station_test.index:
                pos = station_test.index.get_loc(idx)
                if pos + block_len <= len(station_test):
                    block_idx = station_test.index[pos:pos + block_len]
                    factor = rng.uniform(2.5, 5.0)
                    for bidx in block_idx:
                        if bidx not in injected_indices:
                            out.loc[bidx, "n_lends"]   = int(out.loc[bidx, "original_n_lends"] * factor) + rng.randint(1, 5)
                            out.loc[bidx, "n_returns"] = int(out.loc[bidx, "original_n_returns"] * factor) + rng.randint(1, 5)
                            out.loc[bidx, "synth_label"] = 1
                            out.loc[bidx, "synth_type"]  = "collective"
                            injected_indices.add(bidx)

    out["total_demand"] = out["n_lends"] + out["n_returns"]
    out["net_flow"]     = out["n_returns"] - out["n_lends"]
    out["abs_net_flow"] = out["net_flow"].abs()

    if verbose:
        n_inj = out["synth_label"].sum()
        print(f"  Injection: {n_inj:,} Punkte ({n_inj/test_mask.sum()*100:.2f}%)")
        for t, c in out[out["synth_label"]==1]["synth_type"].value_counts().items():
            print(f"    {t}: {c}")
    return out


---\n## NEU in V18: Transfer-spezifische Funktionen

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10 — Transfer Functions: Freeze, Fine-Tune, Data Subsets
# ══════════════════════════════════════════════════════════════
def freeze_ae_encoder(model):
    """Friert Encoder-LSTMs ein. Decoder + Output bleiben trainierbar."""
    for layer in model.layers:
        if "encoder" in layer.name:
            layer.trainable = False
    # Wichtig: nach Trainable-Aenderung neu kompilieren
    trainable = sum(p.numpy().size for p in model.trainable_weights)
    total     = sum(p.numpy().size for p in model.weights)
    print(f"  AE Freeze: {total-trainable:,} frozen, {trainable:,} trainable")
    return model


def freeze_fc_first_lstm(model):
    """Friert erste LSTM-Schicht ein (allgemeine Muster)."""
    for layer in model.layers:
        if layer.name == "forecast_lstm_1":
            layer.trainable = False
    trainable = sum(p.numpy().size for p in model.trainable_weights)
    total     = sum(p.numpy().size for p in model.weights)
    print(f"  FC Freeze: {total-trainable:,} frozen, {trainable:,} trainable")
    return model


def fine_tune_model(model, X_train, Y_train, X_val, Y_val,
                    epochs=30, lr=1e-4, batch_size=512, early_stop=5):
    """Fine-Tuning mit niedrigerer LR und weniger Geduld."""
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0), loss="mse")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=early_stop,
                                      restore_best_weights=True, verbose=1),
    ]
    history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
                        epochs=epochs, batch_size=batch_size,
                        callbacks=callbacks, verbose=1)
    return model, history


def get_last_fraction(X, Y, fraction):
    """Chronologisch letzte `fraction` der Daten (simuliert neue Stadt)."""
    n = len(X)
    start = int(n * (1 - fraction))
    return X[start:], Y[start:]


def get_last_fraction_fc(X_input, Y_target, fraction):
    """Wie get_last_fraction, fuer Forecaster (Input + Target getrennt)."""
    n = len(X_input)
    start = int(n * (1 - fraction))
    return X_input[start:], Y_target[start:]


In [ ]:
# ══════════════════════════════════════════════════════════════
# 11 — V18 Evaluation (per Anomaly-Typ UND Demand-Regime)
# ══════════════════════════════════════════════════════════════
EVAL_ANOMALY_TYPES = ["point_spike", "point_drop", "collective"]
EVAL_REGIMES = ["high", "mid", "low"]

def evaluate_v18(meta, score_col, experiment_name, val_start, test_start, verbose=True):
    """
    Evaluation mit Aufschluesselung nach:
    - Global (PR-AUC, F1, Precision, Recall)
    - Per Anomaly-Typ (point_spike, point_drop, collective)
    - Per Demand-Regime (high, mid, low)
    - Per Typ x Regime (Kreuzmatrix)
    """
    meta = meta.copy()
    meta["split_eval"] = np.where(
        meta["hour_ts"] < val_start, "train",
        np.where(meta["hour_ts"] < test_start, "val", "test"))

    val_m  = meta[meta["split_eval"] == "val"].copy()
    test_m = meta[meta["split_eval"] == "test"].copy()

    if len(val_m) == 0 or len(test_m) == 0:
        print(f"  [{experiment_name}] WARNUNG: Keine VAL/TEST-Daten!")
        return {}

    results = {"experiment": experiment_name, "n_val": len(val_m), "n_test": len(test_m)}

    # --- VAL: Threshold (best F1) ---
    y_val = val_m["synth_label"].astype(int).values
    s_val = val_m[score_col].values
    threshold = None

    if len(np.unique(y_val)) > 1:
        prec_v, rec_v, thr_v = precision_recall_curve(y_val, s_val)
        results["val_pr_auc"] = average_precision_score(y_val, s_val)
        if len(thr_v) > 0:
            f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
            best_idx = np.argmax(f1_v)
            threshold = float(thr_v[best_idx])
            results["val_best_f1"] = float(f1_v[best_idx])
            results["val_threshold"] = threshold

    if threshold is None:
        val_norm = val_m[val_m["synth_label"] == 0][score_col]
        threshold = float(val_norm.quantile(0.99)) if len(val_norm) > 0 else float(test_m[score_col].quantile(0.99))
        results["val_threshold"] = threshold
        results["threshold_source"] = "fallback_99pct"

    # --- TEST: Global ---
    y_test = test_m["synth_label"].astype(int).values
    s_test = test_m[score_col].values
    p_test = (s_test >= threshold).astype(int)

    if len(np.unique(y_test)) > 1:
        results["test_pr_auc"]  = average_precision_score(y_test, s_test)
        results["test_roc_auc"] = roc_auc_score(y_test, s_test)
        results["test_f1"]      = f1_score(y_test, p_test, zero_division=0)
        results["test_prec"]    = precision_score(y_test, p_test, zero_division=0)
        results["test_recall"]  = recall_score(y_test, p_test, zero_division=0)

    # --- TEST: Per Anomaly-Typ ---
    for atype in EVAL_ANOMALY_TYPES:
        type_mask   = test_m["synth_type"] == atype
        normal_mask = test_m["synth_label"] == 0
        n_anom = int(type_mask.sum())
        results[f"n_{atype}"] = n_anom

        if n_anom > 0:
            sub = test_m[type_mask | normal_mask]
            y_t = (sub["synth_type"] == atype).astype(int).values
            y_s = sub[score_col].values
            results[f"pr_{atype}"] = average_precision_score(y_t, y_s) if len(set(y_t))>1 else None
            detected = (test_m.loc[type_mask, score_col] >= threshold).sum()
            results[f"dr_{atype}"] = detected / n_anom
        else:
            results[f"pr_{atype}"] = None
            results[f"dr_{atype}"] = None

    # --- TEST: Per Demand-Regime (global) ---
    for regime in EVAL_REGIMES:
        rm = test_m["demand_regime"] == regime
        n_r = int(rm.sum())
        results[f"n_regime_{regime}"] = n_r
        if n_r > 0:
            y_r = test_m.loc[rm, "synth_label"].astype(int).values
            s_r = test_m.loc[rm, score_col].values
            if len(np.unique(y_r)) > 1:
                results[f"pr_auc_regime_{regime}"] = average_precision_score(y_r, s_r)
                p_r = (s_r >= threshold).astype(int)
                results[f"f1_regime_{regime}"] = f1_score(y_r, p_r, zero_division=0)

    # --- TEST: Kreuzmatrix Typ x Regime ---
    cross_rows = []
    for atype in EVAL_ANOMALY_TYPES:
        for regime in EVAL_REGIMES:
            rm = test_m["demand_regime"] == regime
            tm = test_m["synth_type"] == atype
            nm = test_m["synth_label"] == 0
            n_a = int((rm & tm).sum())
            n_n = int((rm & nm).sum())
            pr, dr = None, None
            if n_a > 0 and n_n > 0:
                sub = test_m[(rm & tm) | (rm & nm)]
                y_t = (sub["synth_type"] == atype).astype(int).values
                y_s = sub[score_col].values
                pr = average_precision_score(y_t, y_s) if len(set(y_t)) > 1 else None
                detected = (test_m.loc[rm & tm, score_col] >= threshold).sum()
                dr = detected / n_a
            cross_rows.append({"type": atype, "regime": regime, "n_anom": n_a, "pr_auc": pr, "dr": dr})
            results[f"pr_{atype}_{regime}"] = pr
            results[f"dr_{atype}_{regime}"] = dr

    results["cross_matrix"] = pd.DataFrame(cross_rows)

    if verbose:
        print(f"\n  [{experiment_name}]")
        print(f"    Threshold: {threshold:.6f}")
        print(f"    TEST Global: PR-AUC={results.get('test_pr_auc','N/A'):.4f}, "
              f"F1={results.get('test_f1','N/A'):.4f}")
        print(f"    Per Type:")
        for at in EVAL_ANOMALY_TYPES:
            pr = results.get(f"pr_{at}")
            dr = results.get(f"dr_{at}")
            n  = results.get(f"n_{at}", 0)
            print(f"      {at:15s} n={n:5d}  PR={pr if pr is None else f'{pr:.4f}':>7s}  "
                  f"DR={dr if dr is None else f'{dr*100:.1f}%':>7s}")
        print(f"    Per Regime:")
        for rg in EVAL_REGIMES:
            pr = results.get(f"pr_auc_regime_{rg}")
            f1 = results.get(f"f1_regime_{rg}")
            n  = results.get(f"n_regime_{rg}", 0)
            print(f"      {rg:6s} n={n:6d}  PR={pr if pr is None else f'{pr:.4f}':>7s}  "
                  f"F1={f1 if f1 is None else f'{f1:.4f}':>7s}")

    return results


In [ ]:
# ══════════════════════════════════════════════════════════════
# 12 — Results Cache (ueberlebt Laufzeit-Unterbrechungen)
# ══════════════════════════════════════════════════════════════
def save_result(results_dict, key, result, cache_path=RESULTS_CACHE_PATH):
    """Speichert einzelnes Experiment-Ergebnis in JSON-Cache."""
    # cross_matrix ist DataFrame -> nicht JSON-serialisierbar
    result_clean = {k: v for k, v in result.items()
                    if k not in ("meta", "cross_matrix") and not isinstance(v, pd.DataFrame)}
    results_dict[key] = result_clean
    with open(cache_path, "w") as f:
        json.dump(results_dict, f, indent=2, default=str)
    return results_dict


def load_results_cache(cache_path=RESULTS_CACHE_PATH):
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            return json.load(f)
    return {}


results_all = load_results_cache()
print(f"Ergebnis-Cache: {len(results_all)} Eintraege geladen")
for k in results_all:
    print(f"  ✓ {k}")


Ergebnis-Cache: 0 Eintraege geladen


In [ ]:
# ══════════════════════════════════════════════════════════════
# 13 — City Data Loading Helper
# ══════════════════════════════════════════════════════════════
def prepare_city_data(demand_path, geo_path, station_names_path, cfg, city_name):
    print(f"\n{'='*70}")
    print(f"  DATEN-PIPELINE: {city_name}")
    print(f"{'='*70}")

    demand = pd.read_parquet(demand_path)
    geo = pd.read_parquet(geo_path)
    snames = pd.read_parquet(station_names_path)

    # Erwartetes Schema prüfen
    demand_required = {
        "location_id", "timestamp", "network_name",
        "station_name_id", "station_id", "n_lends", "n_returns"
    }
    geo_required = {"location_id", "latitude", "longitude"}
    sn_required = {"id", "name"}

    missing_demand = demand_required - set(demand.columns)
    missing_geo = geo_required - set(geo.columns)
    missing_sn = sn_required - set(snames.columns)

    if missing_demand:
        raise ValueError(f"{city_name}: demand fehlt {sorted(missing_demand)}")
    if missing_geo:
        raise ValueError(f"{city_name}: geo fehlt {sorted(missing_geo)}")
    if missing_sn:
        raise ValueError(f"{city_name}: station_names fehlt {sorted(missing_sn)}")

    # Deduplizieren vor Merge
    geo = geo[["location_id", "latitude", "longitude"]].drop_duplicates(subset=["location_id"])
    snames = (
        snames[["id", "name"]]
        .drop_duplicates(subset=["id"])
        .rename(columns={"id": "station_name_id", "name": "station_name"})
    )

    # Typen
    demand["timestamp"] = pd.to_datetime(demand["timestamp"], utc=True, errors="coerce")
    demand = demand.dropna(subset=["timestamp", "station_id"]).copy()

    demand["n_lends"] = pd.to_numeric(demand["n_lends"], errors="coerce").fillna(0).astype(np.int32)
    demand["n_returns"] = pd.to_numeric(demand["n_returns"], errors="coerce").fillna(0).astype(np.int32)

    # Merge Metadaten
    demand = demand.merge(
        snames[["station_name_id", "station_name"]],
        on="station_name_id",
        how="left"
    )
    demand = demand.merge(
        geo[["location_id", "latitude", "longitude"]],
        on="location_id",
        how="left"
    )

    print(f"  Roh: {len(demand):,} Zeilen, {demand['station_id'].nunique()} Stationen")
    print(f"  Zeitraum: {demand['timestamp'].min()} bis {demand['timestamp'].max()}")

    # Aggregation passend zum echten Schema
    df = aggregate_station_level(demand, minutes=cfg.aggregation_minutes)

    # Gap-Fill
    n_before = len(df)
    df = fill_missing_time_bins(df, minutes=cfg.aggregation_minutes)
    print(f"  Gap-Fill: {n_before:,} -> {len(df):,} (+{len(df)-n_before:,})")

    # Split + Labels
    df, station_profile, train_end, val_end = prepare_stations_and_splits(df, cfg, city_name)
    df = run_statistical_pipeline(df, cfg, train_end)

    return df, station_profile, train_end, val_end

In [ ]:
# ══════════════════════════════════════════════════════════════
# 14 — Sequenz-Builder fuer eine Stadt (mit optionalem Scaler)
# ══════════════════════════════════════════════════════════════
ae_features    = cfg.ae_features
score_features = cfg.ae_score_features
context_steps  = cfg.ae_window_size - 1   # 23

def build_city_sequences(df, scaler, ae_features, cfg, city_name="",
                         has_injection=False):
    """Baut Sequenzen fuer eine Stadt. Gibt X, meta, Splits zurueck."""
    df_sorted = df.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    df_sorted = df_sorted.dropna(subset=ae_features)

    df_scaled = df_sorted.copy()
    df_scaled[ae_features] = scaler.transform(df_sorted[ae_features])

    # Labels etc. muessen unskaliert bleiben
    preserve_cols = ["synth_label", "synth_type", "original_n_lends", "original_n_returns",
                     "demand_regime", "label_eval"]
    for col in preserve_cols:
        if col in df_sorted.columns:
            df_scaled[col] = df_sorted[col].values

    # Falls keine Injection: Dummy-Spalten
    if not has_injection:
        if "synth_label" not in df_scaled.columns:
            df_scaled["synth_label"] = 0
            df_scaled["synth_type"]  = "none"

    X, meta = make_sequences_with_window_labels(
        df_scaled, feature_cols=ae_features,
        window_size=cfg.ae_window_size,
        synth_label_col="synth_label", synth_type_col="synth_type",
        require_contiguous=cfg.require_contiguous,
        agg_minutes=cfg.aggregation_minutes)

    print(f"  {city_name}: {X.shape[0]:,} Sequenzen, Shape={X.shape}")
    return X, meta


---\n## Schritt 1: Alle Städte laden

In [ ]:
# ══════════════════════════════════════════════════════════════
# 15 — Alle Staedte laden (Source: clean, Target: clean + injiziert)
# ══════════════════════════════════════════════════════════════

# --- Mannheim (Source, Primary) ---
df_ma, profile_ma, train_end_ma, val_end_ma = prepare_city_data(
    CITY_DEMAND_PATHS["Mannheim"], GEO_PATH, STATION_NAMES_PATH, cfg, "Mannheim")

# --- Heidelberg (Target) ---
df_hd, profile_hd, train_end_hd, val_end_hd = prepare_city_data(
    CITY_DEMAND_PATHS["Heidelberg"], GEO_PATH, STATION_NAMES_PATH, cfg, "Heidelberg")

# --- Giessen (Alt Source) ---
df_gi, profile_gi, train_end_gi, val_end_gi = prepare_city_data(
    CITY_DEMAND_PATHS["Giessen"], GEO_PATH, STATION_NAMES_PATH, cfg, "Giessen")

# --- Kaiserslautern (Multi Source) ---
df_kl, profile_kl, train_end_kl, val_end_kl = prepare_city_data(
    CITY_DEMAND_PATHS["Kaiserslautern"], GEO_PATH, STATION_NAMES_PATH, cfg, "Kaiserslautern")

# Deskriptive Statistik
print("\n" + "="*70)
print("  STAEDTEVERGLEICH")
print("="*70)
for name, df_c, prof in [("Mannheim", df_ma, profile_ma), ("Heidelberg", df_hd, profile_hd),
                          ("Giessen", df_gi, profile_gi), ("Kaiserslautern", df_kl, profile_kl)]:
    n_st = df_c["station_id"].nunique()
    n_days = (df_c["hour_ts"].max() - df_c["hour_ts"].min()).days
    avg_d = df_c["total_demand"].mean()
    print(f"  {name:18s}: {n_st:3d} Stationen, {n_days:4d} Tage, "
          f"avg_demand/h={avg_d:.2f}, Regime={prof['demand_regime'].value_counts().to_dict()}")



  DATEN-PIPELINE: Mannheim
  Roh: 2,547,242 Zeilen, 123 Stationen
  Zeitraum: 2023-03-31 23:00:00+00:00 bis 2026-02-02 23:55:01+00:00
  Gap-Fill: 1,036,178 -> 3,067,251 (+2,031,073)
  Aktive Stationen (Mannheim): 88
  Regime: {'mid': 31, 'high': 31, 'low': 31}
  Zeitraum: 2023-03-31 bis 2026-02-02 (1040 Tage)
  Split: Train < 2025-02-25, Val < 2025-07-30, Test ab 2025-07-30


In [ ]:
# ══════════════════════════════════════════════════════════════
# Injection Cache Helper
# Speichert unter:
# cleaned/demand_injected/<Stadt>/injected_agg{...}_{...}.parquet
# ══════════════════════════════════════════════════════════════
from pathlib import Path

INJECTED_ROOT = Path("cleaned/demand_injected")
INJECTED_ROOT.mkdir(parents=True, exist_ok=True)

def _safe_city_name(city_name: str) -> str:
    return str(city_name).strip().replace("/", "_").replace("\\", "_")

def build_injected_cache_path(city_name, cfg):
    city_dir = INJECTED_ROOT / _safe_city_name(city_name)
    city_dir.mkdir(parents=True, exist_ok=True)

    scenario = str(cfg.injection_scenario)
    rate = str(cfg.injection_rate).replace(".", "p")
    seed = str(cfg.injection_seed)
    agg = str(cfg.aggregation_minutes)

    fname = f"injected_agg{agg}min_{scenario}_rate{rate}_seed{seed}.parquet"
    return city_dir / fname

def load_injected_if_exists(city_name, cfg):
    cache_path = build_injected_cache_path(city_name, cfg)
    if cache_path.exists():
        print(f"  Lade injizierte Datei: {cache_path}")
        df = pd.read_parquet(cache_path)

        if "hour_ts" in df.columns:
            df["hour_ts"] = pd.to_datetime(df["hour_ts"], utc=True, errors="coerce")
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

        return df

    print(f"  Keine injizierte Datei gefunden: {cache_path}")
    return None

def save_injected_dataset(df_inj, city_name, cfg):
    cache_path = build_injected_cache_path(city_name, cfg)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    df_inj.to_parquet(cache_path, index=False)
    print(f"  Injizierte Datei gespeichert unter: {cache_path}")
    return cache_path

def load_or_create_injected_dataset(df_clean, train_end, cfg, city_name, force_reinject=False):
    if not force_reinject:
        cached = load_injected_if_exists(city_name, cfg)
        if cached is not None:
            return cached

    print(f"  Erzeuge neue Injection für {city_name} ...")
    df_inj = inject_synthetic_anomalies(
        df_clean,
        test_start=train_end,
        scenario=cfg.injection_scenario,
        injection_rate=cfg.injection_rate,
        seed=cfg.injection_seed,
        verbose=True
    )

    save_injected_dataset(df_inj, city_name, cfg)
    return df_inj

In [ ]:
# ══════════════════════════════════════════════════════════════
# 16 — Injection NUR auf Heidelberg (mit Laden/Speichern)
# ══════════════════════════════════════════════════════════════
force_reinject = False   # True = neu injizieren und überschreiben

df_hd_inj = load_or_create_injected_dataset(
    df_clean=df_hd,
    train_end=train_end_hd,
    cfg=cfg,
    city_name="Heidelberg",
    force_reinject=force_reinject
)

val_mask_hd = (df_hd_inj["hour_ts"] >= train_end_hd) & (df_hd_inj["hour_ts"] < val_end_hd)
test_mask_hd = df_hd_inj["hour_ts"] >= val_end_hd

print(f"\nInjizierte Anomalien in HD-VAL:  {df_hd_inj.loc[val_mask_hd, 'synth_label'].sum():,}")
print(f"Injizierte Anomalien in HD-TEST: {df_hd_inj.loc[test_mask_hd, 'synth_label'].sum():,}")

for t in EVAL_ANOMALY_TYPES + ["contextual"]:
    n_val = (df_hd_inj.loc[val_mask_hd, "synth_type"] == t).sum()
    n_test = (df_hd_inj.loc[test_mask_hd, "synth_type"] == t).sum()
    print(f"  {t:15s}  VAL={n_val:5d}  TEST={n_test:5d}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 17 — Scaler fitten + Sequenzen bauen (ALLE Staedte)
# ══════════════════════════════════════════════════════════════

# --- Scaler pro Stadt (auf CLEAN Train-Daten) ---
def fit_city_scaler(df_clean, train_end, ae_features, city_name):
    df_c = df_clean.copy().sort_values(["station_id","hour_ts"]).dropna(subset=ae_features)
    train_mask = df_c["hour_ts"] < train_end
    scaler = StandardScaler()
    scaler.fit(df_c.loc[train_mask, ae_features])
    print(f"  Scaler {city_name}: gefittet auf {train_mask.sum():,} Train-Zeilen")
    return scaler

scaler_ma = fit_city_scaler(df_ma, train_end_ma, ae_features, "Mannheim")
scaler_hd = fit_city_scaler(df_hd, train_end_hd, ae_features, "Heidelberg")
scaler_gi = fit_city_scaler(df_gi, train_end_gi, ae_features, "Giessen")
scaler_kl = fit_city_scaler(df_kl, train_end_kl, ae_features, "Kaiserslautern")

# --- Source-Staedte: Sequenzen (CLEAN, kein Inject) ---
print("\n--- Mannheim (Source) ---")
X_ma_all, meta_ma_all = build_city_sequences(df_ma, scaler_ma, ae_features, cfg, "MA", has_injection=False)
meta_ma_all["split_eval"] = np.where(meta_ma_all["hour_ts"] < train_end_ma, "train",
    np.where(meta_ma_all["hour_ts"] < val_end_ma, "val", "test"))
train_normal_ma = (meta_ma_all["split_eval"]=="train") & (meta_ma_all["label_eval"]=="normal")
val_normal_ma   = (meta_ma_all["split_eval"]=="val") & (meta_ma_all["label_eval"]=="normal")
X_ma_train = X_ma_all[train_normal_ma.values]
X_ma_val   = X_ma_all[val_normal_ma.values]
print(f"  MA Train (normal): {X_ma_train.shape}, Val (normal): {X_ma_val.shape}")

# FC Splits fuer MA
score_idx = [ae_features.index(f) for f in score_features]
X_fc_ma_train = X_ma_train[:, :context_steps, :]
Y_fc_ma_train = X_ma_train[:, -1, :][:, score_idx]
X_fc_ma_val   = X_ma_val[:, :context_steps, :]
Y_fc_ma_val   = X_ma_val[:, -1, :][:, score_idx]

# --- Heidelberg (Target): MIT Injection, HD-Scaler ---
print("\n--- Heidelberg (Target, HD-Scaler) ---")
X_hd_all, meta_hd_all = build_city_sequences(df_hd_inj, scaler_hd, ae_features, cfg, "HD", has_injection=True)
meta_hd_all["split_eval"] = np.where(meta_hd_all["hour_ts"] < train_end_hd, "train",
    np.where(meta_hd_all["hour_ts"] < val_end_hd, "val", "test"))

# HD Train/Val: nur normale, CLEAN (fuer Target-Only + Fine-Tune)
train_normal_hd = (meta_hd_all["split_eval"]=="train") & (meta_hd_all["label_eval"]=="normal")
val_normal_hd   = (meta_hd_all["split_eval"]=="val") & (meta_hd_all["synth_label"]==0) & (meta_hd_all["label_eval"]=="normal")

X_hd_train     = X_hd_all[train_normal_hd.values]
X_hd_val_clean = X_hd_all[val_normal_hd.values]
print(f"  HD Train (normal): {X_hd_train.shape}, Val (clean normal): {X_hd_val_clean.shape}")

# FC Splits fuer HD
X_fc_hd_train = X_hd_train[:, :context_steps, :]
Y_fc_hd_train = X_hd_train[:, -1, :][:, score_idx]
X_fc_hd_val   = X_hd_val_clean[:, :context_steps, :]
Y_fc_hd_val   = X_hd_val_clean[:, -1, :][:, score_idx]

# Inference-Daten fuer FC auf HD (alle)
X_fc_hd_input     = X_hd_all[:, :context_steps, :]
X_hd_actual_last   = X_hd_all[:, -1, :]

# --- Heidelberg (Target): MIT Injection, MA-Scaler (NUR fuer Zero-Shot) ---
print("\n--- Heidelberg (Zero-Shot, MA-Scaler) ---")
X_hd_zs_all, meta_hd_zs_all = build_city_sequences(df_hd_inj, scaler_ma, ae_features, cfg, "HD-ZS", has_injection=True)
meta_hd_zs_all["split_eval"] = np.where(meta_hd_zs_all["hour_ts"] < train_end_hd, "train",
    np.where(meta_hd_zs_all["hour_ts"] < val_end_hd, "val", "test"))
X_hd_zs_fc_input = X_hd_zs_all[:, :context_steps, :]

# --- Giessen (Source) ---
print("\n--- Giessen (Source) ---")
X_gi_all, meta_gi_all = build_city_sequences(df_gi, scaler_gi, ae_features, cfg, "GI", has_injection=False)
meta_gi_all["split_eval"] = np.where(meta_gi_all["hour_ts"] < train_end_gi, "train",
    np.where(meta_gi_all["hour_ts"] < val_end_gi, "val", "test"))
train_normal_gi = (meta_gi_all["split_eval"]=="train") & (meta_gi_all["label_eval"]=="normal")
val_normal_gi   = (meta_gi_all["split_eval"]=="val") & (meta_gi_all["label_eval"]=="normal")
X_fc_gi_tr  = X_gi_all[train_normal_gi.values][:, :context_steps, :]
Y_fc_gi_tr  = X_gi_all[train_normal_gi.values][:, -1, :][:, score_idx]
X_fc_gi_val = X_gi_all[val_normal_gi.values][:, :context_steps, :]
Y_fc_gi_val = X_gi_all[val_normal_gi.values][:, -1, :][:, score_idx]

# --- Kaiserslautern (Source) ---
print("\n--- Kaiserslautern (Source) ---")
X_kl_all, meta_kl_all = build_city_sequences(df_kl, scaler_kl, ae_features, cfg, "KL", has_injection=False)
meta_kl_all["split_eval"] = np.where(meta_kl_all["hour_ts"] < train_end_kl, "train",
    np.where(meta_kl_all["hour_ts"] < val_end_kl, "val", "test"))
train_normal_kl = (meta_kl_all["split_eval"]=="train") & (meta_kl_all["label_eval"]=="normal")
val_normal_kl   = (meta_kl_all["split_eval"]=="val") & (meta_kl_all["label_eval"]=="normal")
X_fc_kl_tr  = X_kl_all[train_normal_kl.values][:, :context_steps, :]
Y_fc_kl_tr  = X_kl_all[train_normal_kl.values][:, -1, :][:, score_idx]
X_fc_kl_val = X_kl_all[val_normal_kl.values][:, :context_steps, :]
Y_fc_kl_val = X_kl_all[val_normal_kl.values][:, -1, :][:, score_idx]

# --- Few-Shot Info ---
print("\n" + "="*70)
print("  FEW-SHOT DATENMENGEN (Heidelberg)")
print("="*70)
n_hd_train = len(X_hd_train)
for frac_name, frac in [("1%", 0.01), ("5%", 0.05), ("20%", 0.20)]:
    n = n_hd_train - int(n_hd_train * (1 - frac))
    start_idx = int(n_hd_train * (1 - frac))
    ts_start = meta_hd_all[train_normal_hd].iloc[start_idx]["hour_ts"] if start_idx < len(meta_hd_all[train_normal_hd]) else "N/A"
    print(f"  {frac_name:4s} = {n:6,} Sequenzen (ab ca. {ts_start})")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 18 — V17 Source-Modelle laden (Mannheim)
# ══════════════════════════════════════════════════════════════
print("Lade V17 Source-Modelle (Mannheim)...")
model_ae_ma = keras.models.load_model(V17_AE_PATH)
model_fc_ma = keras.models.load_model(V17_FC_PATH)
print(f"  ✓ AE: {model_ae_ma.count_params():,} Parameter")
print(f"  ✓ FC: {model_fc_ma.count_params():,} Parameter")

# Scaler-Check
with open(V17_SCALER_PATH, "rb") as f:
    scaler_ma_v17 = pickle.load(f)
print(f"  ✓ MA-Scaler geladen (V17)")

# Quick-Verify: MA-Scaler aus V17 vs neu gefittet sollten identisch sein
diff = np.abs(scaler_ma.mean_ - scaler_ma_v17.mean_).max()
print(f"  Scaler-Diff (V17 vs V18): {diff:.8f} {'✓ OK' if diff < 0.01 else '⚠ ABWEICHUNG!'}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 19 — Scaler + Config speichern (fuer Reproduzierbarkeit)
# ══════════════════════════════════════════════════════════════
with open(f"{MODELS_DIR}/scaler_mannheim_v18.pkl", "wb") as f: pickle.dump(scaler_ma, f)
with open(f"{MODELS_DIR}/scaler_heidelberg_v18.pkl", "wb") as f: pickle.dump(scaler_hd, f)
with open(f"{MODELS_DIR}/scaler_giessen_v18.pkl", "wb") as f: pickle.dump(scaler_gi, f)
with open(f"{MODELS_DIR}/scaler_kaiserslautern_v18.pkl", "wb") as f: pickle.dump(scaler_kl, f)

with open(f"{RESULTS_DIR}/v18_config.json", "w") as f:
    json.dump(asdict(cfg), f, indent=2, default=str)

# Splits speichern fuer Re-Entry
splits_info = {
    "MA": {"train_end": str(train_end_ma), "val_end": str(val_end_ma)},
    "HD": {"train_end": str(train_end_hd), "val_end": str(val_end_hd)},
    "GI": {"train_end": str(train_end_gi), "val_end": str(val_end_gi)},
    "KL": {"train_end": str(train_end_kl), "val_end": str(val_end_kl)},
}
with open(f"{RESULTS_DIR}/v18_splits.json", "w") as f:
    json.dump(splits_info, f, indent=2)

print("✓ Scaler, Config, Splits gespeichert.")
print("\n" + "="*70)
print("  SETUP ABGESCHLOSSEN — BEREIT FUER EXPERIMENTE")
print("="*70)


---
## Experiment 1+2 — Transfer-Modus & Datenmenge (Mannheim → Heidelberg)

| Bedingung | AE | FC | Scaler | Beschreibung |
|-----------|----|----|--------|--------------|
| HD Target-Only 1% | ✓ | ✓ | HD | Baseline: Nur 1% HD-Daten, kein Transfer |
| HD Target-Only 100% | ✓ | ✓ | HD | Baseline: Alle HD-Daten, kein Transfer |
| Zero-Shot MA→HD | — | — | **MA** | Source-Modell direkt auf Target, kein Training |
| Fine-Tune 1% MA→HD | ✓ | ✓ | HD | Transfer + 1% HD Fine-Tune |
| Fine-Tune 5% MA→HD | ✓ | ✓ | HD | Transfer + 5% HD Fine-Tune |
| Fine-Tune 20% MA→HD | ✓ | ✓ | HD | Transfer + 20% HD Fine-Tune |

**Scaler-Strategie:** Zero-Shot = MA-Scaler (kein HD-Wissen). Alle anderen = HD-Scaler.
**Freezing:** AE = Encoder frozen. FC = forecast_lstm_1 frozen.


In [ ]:
# ══════════════════════════════════════════════════════════════
# E1a — HD Target-Only 1% (AE + FC) — Baseline
# ══════════════════════════════════════════════════════════════
frac = 0.01
X_hd_tr_1pct, _ = get_last_fraction(X_hd_train, X_hd_train, frac)
X_fc_hd_tr_1pct, Y_fc_hd_tr_1pct = get_last_fraction_fc(X_fc_hd_train, Y_fc_hd_train, frac)
print(f"1% Target-Only: AE={X_hd_tr_1pct.shape}, FC={X_fc_hd_tr_1pct.shape}")

# --- AE ---
def _build_ae_hd_1pct():
    m = build_lstm_autoencoder(len(ae_features), cfg.ae_window_size, cfg.ae_latent_dim,
                               cfg.ae_layers, cfg.ae_dropout, cfg.use_bidirectional, len(ae_features))
    m, h = train_model_generic(m, X_hd_tr_1pct, X_hd_tr_1pct, X_hd_val_clean, X_hd_val_clean,
                               cfg.ae_epochs, cfg.ae_lr, cfg.ae_batch_size, cfg.ae_early_stop)
    return m, h

model_ae_hd_1pct, hist = load_or_train(V18_MODEL_FILES["E1_AE_HD_1pct"], _build_ae_hd_1pct, FORCE_RETRAIN)

X_hat = predict_in_batches(model_ae_hd_1pct, X_hd_all)
meta_hd_all["score_ae_to_1pct"] = compute_reconstruction_scores(X_hd_all, X_hat, ae_features, score_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_ae_to_1pct", train_end_hd, val_end_hd, "score_ae_to_1pct_znorm")
res = evaluate_v18(meta_hd_all, "score_ae_to_1pct_znorm", "E1a_AE_TargetOnly_1pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E1a_AE_TargetOnly_1pct", res)

# --- FC ---
def _build_fc_hd_1pct():
    m = build_lstm_forecaster(len(ae_features), len(score_features), context_steps,
                              cfg.ae_latent_dim, 2, cfg.ae_dropout)
    m, h = train_model_generic(m, X_fc_hd_tr_1pct, Y_fc_hd_tr_1pct, X_fc_hd_val, Y_fc_hd_val,
                               cfg.ae_epochs, cfg.ae_lr, cfg.ae_batch_size, cfg.ae_early_stop)
    return m, h

model_fc_hd_1pct, hist = load_or_train(V18_MODEL_FILES["E1_FC_HD_1pct"], _build_fc_hd_1pct, FORCE_RETRAIN)

meta_hd_all["score_fc_to_1pct"] = compute_forecast_scores(model_fc_hd_1pct, X_fc_hd_input, X_hd_actual_last, score_features, ae_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_fc_to_1pct", train_end_hd, val_end_hd, "score_fc_to_1pct_znorm")
res = evaluate_v18(meta_hd_all, "score_fc_to_1pct_znorm", "E1a_FC_TargetOnly_1pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E1a_FC_TargetOnly_1pct", res)
print("\n✓ E1a abgeschlossen.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# E1b — HD Target-Only 100% (AE + FC) — Upper Bound
# ══════════════════════════════════════════════════════════════
def _build_ae_hd_100pct():
    m = build_lstm_autoencoder(len(ae_features), cfg.ae_window_size, cfg.ae_latent_dim,
                               cfg.ae_layers, cfg.ae_dropout, cfg.use_bidirectional, len(ae_features))
    m, h = train_model_generic(m, X_hd_train, X_hd_train, X_hd_val_clean, X_hd_val_clean,
                               cfg.ae_epochs, cfg.ae_lr, cfg.ae_batch_size, cfg.ae_early_stop)
    return m, h

model_ae_hd_100, hist = load_or_train(V18_MODEL_FILES["E1_AE_HD_100pct"], _build_ae_hd_100pct, FORCE_RETRAIN)

X_hat = predict_in_batches(model_ae_hd_100, X_hd_all)
meta_hd_all["score_ae_to_100"] = compute_reconstruction_scores(X_hd_all, X_hat, ae_features, score_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_ae_to_100", train_end_hd, val_end_hd, "score_ae_to_100_znorm")
res = evaluate_v18(meta_hd_all, "score_ae_to_100_znorm", "E1b_AE_TargetOnly_100pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E1b_AE_TargetOnly_100pct", res)

def _build_fc_hd_100pct():
    m = build_lstm_forecaster(len(ae_features), len(score_features), context_steps,
                              cfg.ae_latent_dim, 2, cfg.ae_dropout)
    m, h = train_model_generic(m, X_fc_hd_train, Y_fc_hd_train, X_fc_hd_val, Y_fc_hd_val,
                               cfg.ae_epochs, cfg.ae_lr, cfg.ae_batch_size, cfg.ae_early_stop)
    return m, h

model_fc_hd_100, hist = load_or_train(V18_MODEL_FILES["E1_FC_HD_100pct"], _build_fc_hd_100pct, FORCE_RETRAIN)

meta_hd_all["score_fc_to_100"] = compute_forecast_scores(model_fc_hd_100, X_fc_hd_input, X_hd_actual_last, score_features, ae_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_fc_to_100", train_end_hd, val_end_hd, "score_fc_to_100_znorm")
res = evaluate_v18(meta_hd_all, "score_fc_to_100_znorm", "E1b_FC_TargetOnly_100pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E1b_FC_TargetOnly_100pct", res)
print("\n✓ E1b abgeschlossen.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# E1c — Zero-Shot MA → HD (kein Training, MA-Scaler!)
# HINWEIS: Z-Norm auf HD-VAL ist der einzige HD-Datenpunkt.
# ══════════════════════════════════════════════════════════════

# AE Zero-Shot
X_hat_zs = predict_in_batches(model_ae_ma, X_hd_zs_all)
meta_hd_zs_all["score_ae_zs"] = compute_reconstruction_scores(X_hd_zs_all, X_hat_zs, ae_features, score_features)
meta_hd_zs_all, _, _ = znorm_score_by_hour(meta_hd_zs_all, "score_ae_zs", train_end_hd, val_end_hd, "score_ae_zs_znorm")
res = evaluate_v18(meta_hd_zs_all, "score_ae_zs_znorm", "E1c_AE_ZeroShot_MA_HD", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E1c_AE_ZeroShot_MA_HD", res)

# FC Zero-Shot
meta_hd_zs_all["score_fc_zs"] = compute_forecast_scores(
    model_fc_ma, X_hd_zs_fc_input, X_hd_zs_all[:, -1, :], score_features, ae_features)
meta_hd_zs_all, _, _ = znorm_score_by_hour(meta_hd_zs_all, "score_fc_zs", train_end_hd, val_end_hd, "score_fc_zs_znorm")
res = evaluate_v18(meta_hd_zs_all, "score_fc_zs_znorm", "E1c_FC_ZeroShot_MA_HD", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E1c_FC_ZeroShot_MA_HD", res)
print("\n✓ E1c (Zero-Shot) abgeschlossen.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# E2a — Fine-Tune 1% MA → HD (AE + FC)
# ══════════════════════════════════════════════════════════════
frac = 0.01
X_hd_ft, _ = get_last_fraction(X_hd_train, X_hd_train, frac)
X_fc_ft, Y_fc_ft = get_last_fraction_fc(X_fc_hd_train, Y_fc_hd_train, frac)
print(f"Fine-Tune {int(frac*100)}%: AE={X_hd_ft.shape}, FC={X_fc_ft.shape}")

# --- AE Fine-Tune ---
def _build_ft_ae_1pct():
    m = keras.models.load_model(V17_AE_PATH)
    m = freeze_ae_encoder(m)
    m, h = fine_tune_model(m, X_hd_ft, X_hd_ft, X_hd_val_clean, X_hd_val_clean,
                           cfg.ft_epochs, cfg.ft_lr, cfg.ft_batch_size, cfg.ft_early_stop)
    return m, h

model_ae_ft, hist = load_or_train(V18_MODEL_FILES["E2_AE_FT_MA_HD_1pct"], _build_ft_ae_1pct, FORCE_RETRAIN)

X_hat = predict_in_batches(model_ae_ft, X_hd_all)
meta_hd_all["score_ae_ft_1pct"] = compute_reconstruction_scores(X_hd_all, X_hat, ae_features, score_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_ae_ft_1pct", train_end_hd, val_end_hd, "score_ae_ft_1pct_znorm")
res = evaluate_v18(meta_hd_all, "score_ae_ft_1pct_znorm", "E2a_AE_FineTune_1pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E2a_AE_FineTune_1pct", res)

# --- FC Fine-Tune ---
def _build_ft_fc_1pct():
    m = keras.models.load_model(V17_FC_PATH)
    m = freeze_fc_first_lstm(m)
    m, h = fine_tune_model(m, X_fc_ft, Y_fc_ft, X_fc_hd_val, Y_fc_hd_val,
                           cfg.ft_epochs, cfg.ft_lr, cfg.ft_batch_size, cfg.ft_early_stop)
    return m, h

model_fc_ft, hist = load_or_train(V18_MODEL_FILES["E2_FC_FT_MA_HD_1pct"], _build_ft_fc_1pct, FORCE_RETRAIN)

meta_hd_all["score_fc_ft_1pct"] = compute_forecast_scores(model_fc_ft, X_fc_hd_input, X_hd_actual_last, score_features, ae_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_fc_ft_1pct", train_end_hd, val_end_hd, "score_fc_ft_1pct_znorm")
res = evaluate_v18(meta_hd_all, "score_fc_ft_1pct_znorm", "E2a_FC_FineTune_1pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E2a_FC_FineTune_1pct", res)
print("\n✓ E2a (Fine-Tune 1%) abgeschlossen.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# E2b — Fine-Tune 5% MA → HD (AE + FC)
# ══════════════════════════════════════════════════════════════
frac = 0.05
X_hd_ft, _ = get_last_fraction(X_hd_train, X_hd_train, frac)
X_fc_ft, Y_fc_ft = get_last_fraction_fc(X_fc_hd_train, Y_fc_hd_train, frac)
print(f"Fine-Tune {int(frac*100)}%: AE={X_hd_ft.shape}, FC={X_fc_ft.shape}")

# --- AE Fine-Tune ---
def _build_ft_ae_5pct():
    m = keras.models.load_model(V17_AE_PATH)
    m = freeze_ae_encoder(m)
    m, h = fine_tune_model(m, X_hd_ft, X_hd_ft, X_hd_val_clean, X_hd_val_clean,
                           cfg.ft_epochs, cfg.ft_lr, cfg.ft_batch_size, cfg.ft_early_stop)
    return m, h

model_ae_ft, hist = load_or_train(V18_MODEL_FILES["E2_AE_FT_MA_HD_5pct"], _build_ft_ae_5pct, FORCE_RETRAIN)

X_hat = predict_in_batches(model_ae_ft, X_hd_all)
meta_hd_all["score_ae_ft_5pct"] = compute_reconstruction_scores(X_hd_all, X_hat, ae_features, score_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_ae_ft_5pct", train_end_hd, val_end_hd, "score_ae_ft_5pct_znorm")
res = evaluate_v18(meta_hd_all, "score_ae_ft_5pct_znorm", "E2b_AE_FineTune_5pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E2b_AE_FineTune_5pct", res)

# --- FC Fine-Tune ---
def _build_ft_fc_5pct():
    m = keras.models.load_model(V17_FC_PATH)
    m = freeze_fc_first_lstm(m)
    m, h = fine_tune_model(m, X_fc_ft, Y_fc_ft, X_fc_hd_val, Y_fc_hd_val,
                           cfg.ft_epochs, cfg.ft_lr, cfg.ft_batch_size, cfg.ft_early_stop)
    return m, h

model_fc_ft, hist = load_or_train(V18_MODEL_FILES["E2_FC_FT_MA_HD_5pct"], _build_ft_fc_5pct, FORCE_RETRAIN)

meta_hd_all["score_fc_ft_5pct"] = compute_forecast_scores(model_fc_ft, X_fc_hd_input, X_hd_actual_last, score_features, ae_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_fc_ft_5pct", train_end_hd, val_end_hd, "score_fc_ft_5pct_znorm")
res = evaluate_v18(meta_hd_all, "score_fc_ft_5pct_znorm", "E2b_FC_FineTune_5pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E2b_FC_FineTune_5pct", res)
print("\n✓ E2b (Fine-Tune 5%) abgeschlossen.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# E2c — Fine-Tune 20% MA → HD (AE + FC)
# ══════════════════════════════════════════════════════════════
frac = 0.2
X_hd_ft, _ = get_last_fraction(X_hd_train, X_hd_train, frac)
X_fc_ft, Y_fc_ft = get_last_fraction_fc(X_fc_hd_train, Y_fc_hd_train, frac)
print(f"Fine-Tune {int(frac*100)}%: AE={X_hd_ft.shape}, FC={X_fc_ft.shape}")

# --- AE Fine-Tune ---
def _build_ft_ae_20pct():
    m = keras.models.load_model(V17_AE_PATH)
    m = freeze_ae_encoder(m)
    m, h = fine_tune_model(m, X_hd_ft, X_hd_ft, X_hd_val_clean, X_hd_val_clean,
                           cfg.ft_epochs, cfg.ft_lr, cfg.ft_batch_size, cfg.ft_early_stop)
    return m, h

model_ae_ft, hist = load_or_train(V18_MODEL_FILES["E2_AE_FT_MA_HD_20pct"], _build_ft_ae_20pct, FORCE_RETRAIN)

X_hat = predict_in_batches(model_ae_ft, X_hd_all)
meta_hd_all["score_ae_ft_20pct"] = compute_reconstruction_scores(X_hd_all, X_hat, ae_features, score_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_ae_ft_20pct", train_end_hd, val_end_hd, "score_ae_ft_20pct_znorm")
res = evaluate_v18(meta_hd_all, "score_ae_ft_20pct_znorm", "E2c_AE_FineTune_20pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E2c_AE_FineTune_20pct", res)

# --- FC Fine-Tune ---
def _build_ft_fc_20pct():
    m = keras.models.load_model(V17_FC_PATH)
    m = freeze_fc_first_lstm(m)
    m, h = fine_tune_model(m, X_fc_ft, Y_fc_ft, X_fc_hd_val, Y_fc_hd_val,
                           cfg.ft_epochs, cfg.ft_lr, cfg.ft_batch_size, cfg.ft_early_stop)
    return m, h

model_fc_ft, hist = load_or_train(V18_MODEL_FILES["E2_FC_FT_MA_HD_20pct"], _build_ft_fc_20pct, FORCE_RETRAIN)

meta_hd_all["score_fc_ft_20pct"] = compute_forecast_scores(model_fc_ft, X_fc_hd_input, X_hd_actual_last, score_features, ae_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_fc_ft_20pct", train_end_hd, val_end_hd, "score_fc_ft_20pct_znorm")
res = evaluate_v18(meta_hd_all, "score_fc_ft_20pct_znorm", "E2c_FC_FineTune_20pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E2c_FC_FineTune_20pct", res)
print("\n✓ E2c (Fine-Tune 20%) abgeschlossen.")


---
## Experiment 3 — Source-City-Wahl: Gießen → Heidelberg (nur FC)
**Frage:** Macht die Source-Stadt einen Unterschied?
Nur LSTM-Forecaster (basierend auf V17-Erkenntnissen, wo FC konsistent besser performte).


In [ ]:
# ══════════════════════════════════════════════════════════════
# E3a — Gießen Source Training (FC)
# ══════════════════════════════════════════════════════════════
def _build_fc_gi():
    m = build_lstm_forecaster(len(ae_features), len(score_features), context_steps,
                              cfg.ae_latent_dim, 2, cfg.ae_dropout)
    m, h = train_model_generic(m, X_fc_gi_tr, Y_fc_gi_tr, X_fc_gi_val, Y_fc_gi_val,
                               cfg.ae_epochs, cfg.ae_lr, cfg.ae_batch_size, cfg.ae_early_stop)
    return m, h

model_fc_gi, hist = load_or_train(V18_MODEL_FILES["E3_FC_GI_source"], _build_fc_gi, FORCE_RETRAIN)
print("✓ Giessen Source-Modell bereit.")

# E3b — Zero-Shot GI → HD
meta_hd_zs_all["score_fc_gi_zs"] = compute_forecast_scores(
    model_fc_gi, X_hd_zs_fc_input, X_hd_zs_all[:, -1, :], score_features, ae_features)
meta_hd_zs_all, _, _ = znorm_score_by_hour(meta_hd_zs_all, "score_fc_gi_zs", train_end_hd, val_end_hd, "score_fc_gi_zs_znorm")
res = evaluate_v18(meta_hd_zs_all, "score_fc_gi_zs_znorm", "E3b_FC_ZeroShot_GI_HD", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E3b_FC_ZeroShot_GI_HD", res)

# E3c — Fine-Tune 1% GI → HD
X_fc_gi_ft, Y_fc_gi_ft = get_last_fraction_fc(X_fc_hd_train, Y_fc_hd_train, 0.01)

def _build_ft_fc_gi_1pct():
    m = keras.models.load_model(V18_MODEL_FILES["E3_FC_GI_source"])
    m = freeze_fc_first_lstm(m)
    m, h = fine_tune_model(m, X_fc_gi_ft, Y_fc_gi_ft, X_fc_hd_val, Y_fc_hd_val,
                           cfg.ft_epochs, cfg.ft_lr, cfg.ft_batch_size, cfg.ft_early_stop)
    return m, h

model_fc_gi_ft, hist = load_or_train(V18_MODEL_FILES["E3_FC_FT_GI_HD_1pct"], _build_ft_fc_gi_1pct, FORCE_RETRAIN)

meta_hd_all["score_fc_gi_ft_1pct"] = compute_forecast_scores(model_fc_gi_ft, X_fc_hd_input, X_hd_actual_last, score_features, ae_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_fc_gi_ft_1pct", train_end_hd, val_end_hd, "score_fc_gi_ft_1pct_znorm")
res = evaluate_v18(meta_hd_all, "score_fc_gi_ft_1pct_znorm", "E3c_FC_FineTune_GI_1pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E3c_FC_FineTune_GI_1pct", res)
print("\n✓ Exp 3 abgeschlossen.")


---
## Experiment 4 — Multi-Source: (MA + GI + KL) → Heidelberg (nur FC)
**Frage:** Verbessert Multi-Source den Transfer?


In [ ]:
# ══════════════════════════════════════════════════════════════
# E4a — Multi-Source Training (MA + GI + KL → FC)
# ══════════════════════════════════════════════════════════════
X_fc_multi_tr = np.concatenate([X_fc_ma_train, X_fc_gi_tr, X_fc_kl_tr], axis=0)
Y_fc_multi_tr = np.concatenate([Y_fc_ma_train, Y_fc_gi_tr, Y_fc_kl_tr], axis=0)
X_fc_multi_val = np.concatenate([X_fc_ma_val, X_fc_gi_val, X_fc_kl_val], axis=0)
Y_fc_multi_val = np.concatenate([Y_fc_ma_val, Y_fc_gi_val, Y_fc_kl_val], axis=0)
print(f"Multi-Source Train: {X_fc_multi_tr.shape}")
print(f"Multi-Source Val:   {X_fc_multi_val.shape}")

def _build_fc_multi():
    m = build_lstm_forecaster(len(ae_features), len(score_features), context_steps,
                              cfg.ae_latent_dim, 2, cfg.ae_dropout)
    m, h = train_model_generic(m, X_fc_multi_tr, Y_fc_multi_tr, X_fc_multi_val, Y_fc_multi_val,
                               cfg.ae_epochs, cfg.ae_lr, cfg.ae_batch_size, cfg.ae_early_stop)
    return m, h

model_fc_multi, hist = load_or_train(V18_MODEL_FILES["E4_FC_multi_source"], _build_fc_multi, FORCE_RETRAIN)

# E4b — Multi FT 1%
X_fc_m_ft1, Y_fc_m_ft1 = get_last_fraction_fc(X_fc_hd_train, Y_fc_hd_train, 0.01)
def _build_ft_multi_1pct():
    m = keras.models.load_model(V18_MODEL_FILES["E4_FC_multi_source"])
    m = freeze_fc_first_lstm(m)
    m, h = fine_tune_model(m, X_fc_m_ft1, Y_fc_m_ft1, X_fc_hd_val, Y_fc_hd_val,
                           cfg.ft_epochs, cfg.ft_lr, cfg.ft_batch_size, cfg.ft_early_stop)
    return m, h

model_fc_m_ft1, hist = load_or_train(V18_MODEL_FILES["E4_FC_FT_multi_1pct"], _build_ft_multi_1pct, FORCE_RETRAIN)
meta_hd_all["score_fc_multi_ft1"] = compute_forecast_scores(model_fc_m_ft1, X_fc_hd_input, X_hd_actual_last, score_features, ae_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_fc_multi_ft1", train_end_hd, val_end_hd, "score_fc_multi_ft1_znorm")
res = evaluate_v18(meta_hd_all, "score_fc_multi_ft1_znorm", "E4b_FC_Multi_FT_1pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E4b_FC_Multi_FT_1pct", res)

# E4c — Multi FT 10%
X_fc_m_ft10, Y_fc_m_ft10 = get_last_fraction_fc(X_fc_hd_train, Y_fc_hd_train, 0.10)
def _build_ft_multi_10pct():
    m = keras.models.load_model(V18_MODEL_FILES["E4_FC_multi_source"])
    m = freeze_fc_first_lstm(m)
    m, h = fine_tune_model(m, X_fc_m_ft10, Y_fc_m_ft10, X_fc_hd_val, Y_fc_hd_val,
                           cfg.ft_epochs, cfg.ft_lr, cfg.ft_batch_size, cfg.ft_early_stop)
    return m, h

model_fc_m_ft10, hist = load_or_train(V18_MODEL_FILES["E4_FC_FT_multi_10pct"], _build_ft_multi_10pct, FORCE_RETRAIN)
meta_hd_all["score_fc_multi_ft10"] = compute_forecast_scores(model_fc_m_ft10, X_fc_hd_input, X_hd_actual_last, score_features, ae_features)
meta_hd_all, _, _ = znorm_score_by_hour(meta_hd_all, "score_fc_multi_ft10", train_end_hd, val_end_hd, "score_fc_multi_ft10_znorm")
res = evaluate_v18(meta_hd_all, "score_fc_multi_ft10_znorm", "E4c_FC_Multi_FT_10pct", train_end_hd, val_end_hd)
results_all = save_result(results_all, "E4c_FC_Multi_FT_10pct", res)
print("\n✓ Exp 4 abgeschlossen.")
print("\n" + "="*60 + "\n  ALLE EXPERIMENTE ABGESCHLOSSEN!\n" + "="*60)


---\n## Gesamtvergleich

In [ ]:
# ══════════════════════════════════════════════════════════════
# Ergebnistabelle
# ══════════════════════════════════════════════════════════════
results_all = load_results_cache()

EXP_ORDER = [
    ("E1a_AE_TargetOnly_1pct",   "AE","Target-Only","HD","1%"),
    ("E1b_AE_TargetOnly_100pct", "AE","Target-Only","HD","100%"),
    ("E1c_AE_ZeroShot_MA_HD",    "AE","Zero-Shot","MA","0%"),
    ("E2a_AE_FineTune_1pct",     "AE","Fine-Tune","MA","1%"),
    ("E2b_AE_FineTune_5pct",     "AE","Fine-Tune","MA","5%"),
    ("E2c_AE_FineTune_20pct",    "AE","Fine-Tune","MA","20%"),
    ("E1a_FC_TargetOnly_1pct",   "FC","Target-Only","HD","1%"),
    ("E1b_FC_TargetOnly_100pct", "FC","Target-Only","HD","100%"),
    ("E1c_FC_ZeroShot_MA_HD",    "FC","Zero-Shot","MA","0%"),
    ("E2a_FC_FineTune_1pct",     "FC","Fine-Tune","MA","1%"),
    ("E2b_FC_FineTune_5pct",     "FC","Fine-Tune","MA","5%"),
    ("E2c_FC_FineTune_20pct",    "FC","Fine-Tune","MA","20%"),
    ("E3b_FC_ZeroShot_GI_HD",    "FC","Zero-Shot","GI","0%"),
    ("E3c_FC_FineTune_GI_1pct",  "FC","Fine-Tune","GI","1%"),
    ("E4b_FC_Multi_FT_1pct",     "FC","Multi-FT","MA+GI+KL","1%"),
    ("E4c_FC_Multi_FT_10pct",    "FC","Multi-FT","MA+GI+KL","10%"),
]

rows = []
for key, mdl, mode, src, tgt_pct in EXP_ORDER:
    r = results_all.get(key, {})
    row = {"Experiment": key, "Modell": mdl, "Modus": mode, "Source": src, "Target%": tgt_pct,
           "PR-AUC": r.get("test_pr_auc"), "F1": r.get("test_f1")}
    for at in EVAL_ANOMALY_TYPES:
        row[f"PR_{at}"] = r.get(f"pr_{at}")
        row[f"DR_{at}"] = r.get(f"dr_{at}")
    for rg in EVAL_REGIMES:
        row[f"PR_{rg}"] = r.get(f"pr_auc_regime_{rg}")
        row[f"F1_{rg}"] = r.get(f"f1_regime_{rg}")
    rows.append(row)

df_compare = pd.DataFrame(rows)
df_compare.to_csv(f"{RESULTS_DIR}/v18_transfer_comparison.csv", index=False)
df_compare.to_parquet(f"{RESULTS_DIR}/v18_transfer_comparison.parquet")

# Print
def fmt(v): return f"{v:.4f}" if v is not None and not (isinstance(v,float) and np.isnan(v)) else "N/A"

print("="*130)
print("  V18 TRANSFER LEARNING — GESAMTVERGLEICH")
print("="*130)
for _, row in df_compare.iterrows():
    print(f"{row['Experiment']:<35} {row['Modell']:3} {row['Modus']:<12} {row['Source']:<10} {row['Target%']:5} "
          f"PR-AUC={fmt(row['PR-AUC'])} F1={fmt(row['F1'])} "
          f"| Spk={fmt(row.get('PR_point_spike'))} Drp={fmt(row.get('PR_point_drop'))} Coll={fmt(row.get('PR_collective'))} "
          f"| H={fmt(row.get('PR_high'))} M={fmt(row.get('PR_mid'))} L={fmt(row.get('PR_low'))}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Alle Artefakte speichern
# ══════════════════════════════════════════════════════════════
# Meta mit allen Scores
score_cols = [c for c in meta_hd_all.columns if c.startswith("score_")]
meta_save = meta_hd_all[["station_id","hour_ts","split_eval","synth_label","synth_type",
                          "demand_regime"] + score_cols].copy()
meta_save.to_parquet(f"{RESULTS_DIR}/meta_hd_all_v18.parquet")

with open(f"{RESULTS_DIR}/v18_all_results.json", "w") as f:
    json.dump(results_all, f, indent=2, default=str)

print("="*60)
print("  V18 ARTEFAKTE GESPEICHERT")
print("="*60)
for f_path in sorted(glob.glob(f"{RESULTS_DIR}/*") + glob.glob(f"{MODELS_DIR}/*")):
    sz = os.path.getsize(f_path) / 1e6
    print(f"  {'✓' if sz > 0 else '?'} {os.path.basename(f_path):45s} {sz:6.1f} MB")
